In [26]:
import pandas as pd
import numpy as np
from datetime import datetime
import duckdb as db

In [27]:
contratos = pd.read_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_contratos.csv")
sancoes = pd.read_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_sancoes.csv", sep=";")
indices = pd.read_csv("/Users/lucasborges/Desktop/supply_chain/data/bronze/raw_indices.csv", sep=";")

In [28]:
# Dimensões

print("Dimensão Contratos:", contratos.columns)
print()
print("Dimensão Sanções:", sancoes.columns)
print()
print("Dimensão Índices:", indices.columns)

Dimensão Contratos: Index(['numeroControlePncpCompra', 'codigoPaisFornecedor', 'dataAtualizacao',
       'orgaoEntidade', 'dataAssinatura', 'dataVigenciaInicio',
       'dataVigenciaFim', 'niFornecedor', 'tipoPessoa', 'processo',
       'orgaoSubRogado', 'unidadeOrgao', 'unidadeSubRogada',
       'nomeRazaoSocialFornecedor', 'informacaoComplementar',
       'numeroContratoEmpenho', 'categoriaProcesso', 'anoContrato',
       'tipoContrato', 'sequencialContrato', 'dataPublicacaoPncp',
       'niFornecedorSubContratado', 'nomeFornecedorSubContratado',
       'dataAtualizacaoGlobal', 'numeroControlePNCP', 'receita',
       'numeroParcelas', 'numeroRetificacao', 'tipoPessoaSubContratada',
       'objetoContrato', 'valorInicial', 'valorParcela', 'valorGlobal',
       'valorAcumulado', 'identificadorCipi', 'urlCipi', 'usuarioNome'],
      dtype='object')

Dimensão Sanções: Index(['CNPJ/CPF da Pessoa ou Empresa Sancionada',
       'Nome da Pessoa ou Empresa Sancionada', 'Cadastro', 'UF sancion

In [29]:
# Renomear colunas
map_contratos = {'niFornecedor': 'cnpj',
            'nomeRazaoSocialFornecedor': 'razao_social'}

map_sancoes = {'CNPJ/CPF da Pessoa ou Empresa Sancionada': 'cnpj',
           'Nome da Pessoa ou Empresa Sancionada': 'razao_social'}

contratos.rename(columns=map_contratos, inplace=True)
sancoes.rename(columns=map_sancoes, inplace=True)

# Padronização caracteres
contratos['cnpj'] = contratos['cnpj'].astype(str).str.replace(r'[^\w]', '', regex=True)
sancoes['cnpj'] = sancoes['cnpj'].astype(str).str.replace(r'[^\w]', '', regex=True)

print("Colunas Contratos:", contratos.columns)
print("Colunas Sanções:", sancoes.columns)

Colunas Contratos: Index(['numeroControlePncpCompra', 'codigoPaisFornecedor', 'dataAtualizacao',
       'orgaoEntidade', 'dataAssinatura', 'dataVigenciaInicio',
       'dataVigenciaFim', 'cnpj', 'tipoPessoa', 'processo', 'orgaoSubRogado',
       'unidadeOrgao', 'unidadeSubRogada', 'razao_social',
       'informacaoComplementar', 'numeroContratoEmpenho', 'categoriaProcesso',
       'anoContrato', 'tipoContrato', 'sequencialContrato',
       'dataPublicacaoPncp', 'niFornecedorSubContratado',
       'nomeFornecedorSubContratado', 'dataAtualizacaoGlobal',
       'numeroControlePNCP', 'receita', 'numeroParcelas', 'numeroRetificacao',
       'tipoPessoaSubContratada', 'objetoContrato', 'valorInicial',
       'valorParcela', 'valorGlobal', 'valorAcumulado', 'identificadorCipi',
       'urlCipi', 'usuarioNome'],
      dtype='object')
Colunas Sanções: Index(['cnpj', 'razao_social', 'Cadastro', 'UF sancionado',
       'Nome do Órgão Sancionador', 'Categoria sanção', 'Data Publicação',
       'Va

In [30]:
# Lista de colunas para MANTER 
cols_contratos_keep = [
    'dataAssinatura', 'dataVigenciaInicio', 'dataVigenciaFim', 
    'cnpj', 'razao_social', 'unidadeOrgao', 
    'objetoContrato', 'valorGlobal', 'tipoContrato'
]

cols_sancoes_keep = [
    'cnpj', 'razao_social', 'Cadastro', 
    'Nome do Órgão Sancionador', 'Categoria sanção', 'Data Publicação'
]

# Filtrando os DataFrames
contratos = contratos[cols_contratos_keep].copy()
sancoes = sancoes[cols_sancoes_keep].copy()

# Visualizando para garantir
print(f"Contratos shape: {contratos.shape}")
print(f"Sancoes shape: {sancoes.shape}")

Contratos shape: (122, 9)
Sancoes shape: (5485, 6)


In [31]:
# Contagem duplicatas
print('Quantiade de valores duplicados em sanções:', sancoes.duplicated().sum())
print('Quantidade de valores duplicados em contratos:', contratos.duplicated().sum())

Quantiade de valores duplicados em sanções: 185
Quantidade de valores duplicados em contratos: 0


In [32]:
# Remoção duplicatas
sancoes = sancoes.drop_duplicates(keep='first')

In [33]:
contratos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   dataAssinatura      122 non-null    object 
 1   dataVigenciaInicio  122 non-null    object 
 2   dataVigenciaFim     122 non-null    object 
 3   cnpj                122 non-null    object 
 4   razao_social        122 non-null    object 
 5   unidadeOrgao        122 non-null    object 
 6   objetoContrato      122 non-null    object 
 7   valorGlobal         122 non-null    float64
 8   tipoContrato        122 non-null    object 
dtypes: float64(1), object(8)
memory usage: 8.7+ KB


In [34]:
sancoes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5300 entries, 0 to 5484
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   cnpj                       5300 non-null   object
 1   razao_social               5300 non-null   object
 2   Cadastro                   5300 non-null   object
 3   Nome do Órgão Sancionador  5300 non-null   object
 4   Categoria sanção           5300 non-null   object
 5   Data Publicação            5300 non-null   object
dtypes: object(6)
memory usage: 289.8+ KB


In [35]:
sancoes["Data Publicação"] = pd.to_datetime(sancoes["Data Publicação"], format="%d/%m/%Y", errors='coerce')

In [36]:
contratos['dataAssinatura'] = pd.to_datetime(contratos['dataAssinatura'])
contratos['dataVigenciaInicio'] = pd.to_datetime(contratos['dataVigenciaInicio'])
contratos['dataVigenciaFim'] = pd.to_datetime(contratos['dataVigenciaFim'])

In [37]:
contratos.head()

,dataAssinatura,dataVigenciaInicio,dataVigenciaFim,cnpj,razao_social,unidadeOrgao,objetoContrato,valorGlobal,tipoContrato
0,2023-01-24,2023-01-24,2023-12-31,10719671000160,ELDEX DISTRIBUIDORA DE JORNAIS E REVISTAS LTDA,"{'ufNome': 'Distrito Federal', 'codigoUnidade'...",PRESTAÇÃO DE SERVIÇOS DE FORNECIMENTO DE ASSIN...,10575.00,"{'id': 7, 'nome': 'Empenho'}"
1,2023-01-24,2023-01-24,2023-12-31,72579105000158,JOSE DOS REIS CHAVEIRO,"{'ufNome': 'Distrito Federal', 'codigoUnidade'...","SERVIÇOS DE CHAVEIRO, COM FORNECIMENTO DE MATE...",4000.00,"{'id': 7, 'nome': 'Empenho'}"
2,2023-01-24,2023-01-24,2023-12-31,01319181000186,LAVANDERIA CRISTAL SERVICOS EXPRESSOS LTDA,"{'ufNome': 'Distrito Federal', 'codigoUnidade'...","SERVIÇOS DE LAVANDERIA, LAVAGEM, SECAGEM, PASS...",2000.00,"{'id': 7, 'nome': 'Empenho'}"
3,2023-01-24,2023-01-24,2023-12-31,24212365000148,EDMAR FERREIRA DA SILVA,"{'ufNome': 'Distrito Federal', 'codigoUnidade'...","SERVIÇOS DE DESINSETIZAÇÃO, DESRATIZAÇÃO, DESC...",3909.99,"{'id': 7, 'nome': 'Empenho'}"
4,2023-04-27,2023-04-27,2023-12-31,17067013000180,EGP SERVICOS LTDA,"{'ufNome': 'Distrito Federal', 'codigoUnidade'...",CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA CONF...,3198.00,"{'id': 7, 'nome': 'Empenho'}"


In [38]:
contratos['tipoContrato'].value_counts()

tipoContrato
{'id': 1, 'nome': 'Contrato (termo inicial)'}    62
{'id': 7, 'nome': 'Empenho'}                     59
{'id': 4, 'nome': 'Concessão'}                    1
Name: count, dtype: int64

In [39]:
contratos['unidadeOrgao'].value_counts()  



unidadeOrgao
{'ufNome': 'Distrito Federal', 'codigoUnidade': '320004', 'ufSigla': 'DF', 'municipioNome': 'Brasília', 'nomeUnidade': 'MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF', 'codigoIbge': '5300108'}    94
{'ufNome': 'Distrito Federal', 'codigoUnidade': '320005', 'ufSigla': 'DF', 'municipioNome': 'Brasília', 'nomeUnidade': 'COORD.GERAL DE RECURSOS HUMANOS/M.M.E', 'codigoIbge': '5300108'}            23
{'ufNome': 'Distrito Federal', 'codigoUnidade': '320017', 'ufSigla': 'DF', 'municipioNome': 'Brasília', 'nomeUnidade': 'SEC. DE PETROLEO,GAS NATURAL E COMBUS. RENOV.', 'codigoIbge': '5300108'}     4
{'ufNome': 'Distrito Federal', 'codigoUnidade': '320084', 'ufSigla': 'DF', 'municipioNome': 'Brasília', 'nomeUnidade': 'SUBSECRETARIA DE TECNOLOGIA E INOVAÇAO/MME', 'codigoIbge': '5300108'}        1
Name: count, dtype: int64

In [40]:
# Transformando as colunas de dicionários em strings legíveis

contratos['unidadeOrgao'] = contratos['unidadeOrgao'].str.extract(r"'nomeUnidade': '([^']+)'")
contratos['tipoContrato'] = contratos['tipoContrato'].str.extract(r"'nome': '([^']+)'")

In [41]:
contratos

,dataAssinatura,dataVigenciaInicio,dataVigenciaFim,cnpj,razao_social,unidadeOrgao,objetoContrato,valorGlobal,tipoContrato
0,2023-01-24,2023-01-24,2023-12-31,10719671000160,ELDEX DISTRIBUIDORA DE JORNAIS E REVISTAS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,PRESTAÇÃO DE SERVIÇOS DE FORNECIMENTO DE ASSIN...,10575.00,Empenho
1,2023-01-24,2023-01-24,2023-12-31,72579105000158,JOSE DOS REIS CHAVEIRO,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,"SERVIÇOS DE CHAVEIRO, COM FORNECIMENTO DE MATE...",4000.00,Empenho
2,2023-01-24,2023-01-24,2023-12-31,01319181000186,LAVANDERIA CRISTAL SERVICOS EXPRESSOS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,"SERVIÇOS DE LAVANDERIA, LAVAGEM, SECAGEM, PASS...",2000.00,Empenho
3,2023-01-24,2023-01-24,2023-12-31,24212365000148,EDMAR FERREIRA DA SILVA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,"SERVIÇOS DE DESINSETIZAÇÃO, DESRATIZAÇÃO, DESC...",3909.99,Empenho
4,2023-04-27,2023-04-27,2023-12-31,17067013000180,EGP SERVICOS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA CONF...,3198.00,Empenho
...,...,...,...,...,...,...,...,...,...
117,2025-10-28,2025-10-28,2027-10-28,51311014000167,LAG LICENCIAMENTOS DE SOFTWARES LTDA,SUBSECRETARIA DE TECNOLOGIA E INOVAÇAO/MME,"ASSINATURA CORPORATIVA DO SERVIÇO ON-LINE ""LIS...",4784.00,Empenho
118,2025-11-28,2025-12-01,2025-12-05,04451208000188,COBUCCI DESENVOLVIMENTO HUMANO LTDA,COORD.GERAL DE RECURSOS HUMANOS/M.M.E,CURSO SOBRE “MEDIAÇÃO DE CONFLITOS E TÉCNICA D...,17210.00,Empenho
119,2025-12-23,2025-12-23,2028-12-23,59104760000604,TOYOTA DO BRASIL LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,AQUISIÇÃO DE 2 (DOIS) VEÍCULOS AUTOMOTORES NOV...,425429.00,Contrato (termo inicial)
120,2025-12-02,2026-01-01,2027-01-01,28993675000106,O2 AMBIENTAL LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA PRES...,94080.00,Contrato (termo inicial)


In [42]:
import pandas as pd
import re

def categorizar_objeto(texto):
    # 1. Normalização: tudo minúsculo para facilitar o match
    if pd.isna(texto): return "NAO INFORMADO"
    t = str(texto).lower()
    
    # 2. Dicionário de Regras
    
    # --- FACILITIES & MANUTENÇÃO ---
    if any(x in t for x in ['elevador', 'ar condicionado', 'climatização', 'refrigeração']):
        return 'Manutenção de Equipamentos'
    if any(x in t for x in ['limpeza', 'conservação', 'dedetização', 'desinsetização', 'pragas', 'jardinagem']):
        return 'Facilities - Limpeza/Zeladoria'
    if any(x in t for x in ['instalação', 'chaveiro', 'persiana', 'cortina', 'lavanderia', 'resíduos', 'lixo']):
        return 'Facilities - Serviços Gerais'
    if any(x in t for x in ['gás', 'glp', 'obra', 'engenharia', 'reforma']):
        return 'Engenharia e Obras'
        
    # --- FROTA & LOGÍSTICA ---
    if any(x in t for x in ['locação de veículo', 'transporte', 'motorista']):
        return 'Logística - Locação/Serviço'
    if any(x in t for x in ['aquisição de veículo', 'automóvel', 'carro', 'frotas']):
        return 'Logística - Aquisição de Frota'
    
    # --- TI & INFORMAÇÃO ---
    if any(x in t for x in ['software', 'licença', 'desktop' 'certificado digital', 'assinatura eletrônica', 'site', 'hospedagem']):
        return 'TI & Software'
    if any(x in t for x in ['jornal', 'notícia', 'broadcast', 'agência estado', 'clipping']):
        return 'Serviços de Informação'
        
    # --- ADMINISTRATIVO ---
    if any(x in t for x in ['evento', 'coffee', 'lanche', 'alimentação', 'g20', 'congresso']):
        return 'Eventos e Alimentação'
    if any(x in t for x in ['tradução', 'assessoria', 'interpretação', 'curso', 'treinamento', 'palestra', 'seminário']):
        return 'Serviços Profissionais'
    if any(x in t for x in ['cartão de visita', 'notebooks', 'impressão', 'gráfica', 'mobiliário']):
        return 'Material de Escritório/Marketing'
        
    # 3. Categoria Residual
    return 'Outros / Serviços Gerais'

# Aplicação no DataFrame
contratos['categoria_spend'] = contratos['objetoContrato'].apply(categorizar_objeto)

In [43]:
# ============================================
# CAMADA SILVER - Salvar DataFrames limpos
# ============================================

contratos.to_parquet("/Users/lucasborges/Desktop/supply_chain/data/silver/contratos.parquet", index=False)
sancoes.to_parquet("/Users/lucasborges/Desktop/supply_chain/data/silver/sancoes.parquet", index=False)

print("Arquivos Silver salvos:")
print("  - data/silver/contratos.parquet")
print("  - data/silver/sancoes.parquet")

Arquivos Silver salvos:
  - data/silver/contratos.parquet
  - data/silver/sancoes.parquet


In [44]:
# ============================================
# CAMADA GOLD - Banco DuckDB com Views SQL
# ============================================

import duckdb

# Criar conexão persistente
con = duckdb.connect("/Users/lucasborges/Desktop/supply_chain/data/gold/supply_chain.duckdb")

# Registrar DataFrames como tabelas
con.execute("CREATE OR REPLACE TABLE contratos AS SELECT * FROM contratos")
con.execute("CREATE OR REPLACE TABLE sancoes AS SELECT * FROM sancoes")

print("Tabelas criadas: contratos, sancoes")

Tabelas criadas: contratos, sancoes


In [ ]:
# ============================================
# VIEWS CONSOLIDADAS PARA STREAMLIT
# ============================================

# View 1: Contratos base
con.execute("""
    CREATE OR REPLACE VIEW vw_contratos AS
    SELECT * FROM contratos
""")

# View 2: Sanções base
con.execute("""
    CREATE OR REPLACE VIEW vw_sancoes AS
    SELECT * FROM sancoes
""")

# View 3: Contratos com flag de risco (fornecedor sancionado)
con.execute("""
    CREATE OR REPLACE VIEW vw_contratos_risco AS
    WITH sancoes_agrupadas AS (
        SELECT 
            cnpj, 
            STRING_AGG("Categoria sanção", ', ') AS lista_sancoes,
            STRING_AGG("Nome do Órgão Sancionador", ', ') AS lista_orgaos
        FROM sancoes
        GROUP BY cnpj
    )
    SELECT 
        c.*,
        CASE WHEN s.cnpj IS NOT NULL THEN true ELSE false END AS tem_sancao,
        s.lista_sancoes,
        s.lista_orgaos
    FROM contratos c
    LEFT JOIN sancoes_agrupadas s ON c.cnpj = s.cnpj
""")

# View 4: Resumo por categoria de spend
con.execute("""
    CREATE OR REPLACE VIEW vw_resumo_categoria_spend AS
    SELECT 
        categoria_spend,
        COUNT(*) AS qtd_contratos,
        SUM(valorGlobal) AS valor_total,
        AVG(valorGlobal) AS valor_medio
    FROM contratos
    GROUP BY categoria_spend
    ORDER BY valor_total DESC
""")

# View 5: Resumo mensal
con.execute("""
    CREATE OR REPLACE VIEW vw_resumo_mensal AS
    SELECT 
        DATE_TRUNC('month', dataAssinatura) AS mes,
        COUNT(*) AS qtd_contratos,
        SUM(valorGlobal) AS valor_total
    FROM contratos
    GROUP BY DATE_TRUNC('month', dataAssinatura)
    ORDER BY mes
""")

# View 6: KPIs gerais
con.execute("""
    CREATE OR REPLACE VIEW vw_kpis AS
    SELECT 
        COUNT(*) AS total_contratos,
        SUM(valorGlobal) AS valor_total_contratos,
        COUNT(DISTINCT cnpj) AS total_fornecedores,
        (SELECT COUNT(DISTINCT c.cnpj) 
         FROM contratos c 
         INNER JOIN sancoes s ON c.cnpj = s.cnpj) AS fornecedores_com_sancao,
        ROUND(
            (SELECT COUNT(DISTINCT c.cnpj) FROM contratos c INNER JOIN sancoes s ON c.cnpj = s.cnpj) * 100.0 
            / NULLIF(COUNT(DISTINCT cnpj), 0), 2
        ) AS percentual_risco
    FROM contratos
""")

# View 7: Índice HHI de Concentração de Mercado (D02)
# HHI = Somatório do quadrado do Market Share de cada fornecedor
# HHI < 1500: Competitivo | 1500-2500: Moderado | > 2500: Altamente Concentrado
con.execute("""
    CREATE OR REPLACE VIEW vw_concentracao_hhi AS
    WITH total_categoria AS (
        SELECT categoria_spend, SUM(valorGlobal) AS total_cat
        FROM contratos
        GROUP BY categoria_spend
    ),
    share_fornecedor AS (
        SELECT 
            c.categoria_spend,
            c.razao_social,
            SUM(c.valorGlobal) AS total_fornecedor,
            (SUM(c.valorGlobal) / tc.total_cat) * 100 AS market_share
        FROM contratos c
        JOIN total_categoria tc ON c.categoria_spend = tc.categoria_spend
        GROUP BY c.categoria_spend, c.razao_social, tc.total_cat
    )
    SELECT 
        categoria_spend,
        ROUND(SUM(market_share * market_share), 2) AS hhi_index,
        COUNT(DISTINCT razao_social) AS qtd_fornecedores,
        CASE 
            WHEN SUM(market_share * market_share) > 2500 THEN 'Crítica (Monopólio)'
            WHEN SUM(market_share * market_share) > 1500 THEN 'Moderada'
            ELSE 'Baixa (Competitiva)'
        END AS classificacao_risco
    FROM share_fornecedor
    GROUP BY categoria_spend
    ORDER BY hhi_index DESC
""")

# View 8: Detecção de Anomalias de Preço - Z-Score (I02)
# Identifica contratos com valores significativamente acima/abaixo da média da categoria
con.execute("""
    CREATE OR REPLACE VIEW vw_anomalias_preco AS
    WITH estatisticas AS (
        SELECT 
            categoria_spend,
            AVG(valorGlobal) AS media_cat,
            STDDEV(valorGlobal) AS desvio_cat
        FROM contratos
        GROUP BY categoria_spend
    )
    SELECT 
        c.cnpj,
        c.razao_social,
        c.objetoContrato,
        c.valorGlobal,
        c.categoria_spend,
        c.dataAssinatura,
        ROUND(e.media_cat, 2) AS media_categoria,
        ROUND(e.desvio_cat, 2) AS desvio_categoria,
        ROUND((c.valorGlobal - e.media_cat) / NULLIF(e.desvio_cat, 0), 2) AS z_score,
        CASE 
            WHEN e.desvio_cat IS NULL OR e.desvio_cat = 0 THEN 'Categoria com único valor'
            WHEN (c.valorGlobal - e.media_cat) > (2 * e.desvio_cat) THEN 'Alto Valor (Anômalo)'
            WHEN (c.valorGlobal - e.media_cat) < (-1.5 * e.desvio_cat) THEN 'Suspeita Subpreço'
            ELSE 'Dentro da Normalidade'
        END AS status_preco
    FROM contratos c
    JOIN estatisticas e ON c.categoria_spend = e.categoria_spend
    ORDER BY z_score DESC
""")

# View 9: Radar de Vencimentos - Alertas de Contratos (D03)
# Identifica contratos próximos do vencimento para planejamento de renovações
con.execute("""
    CREATE OR REPLACE VIEW vw_alerta_vencimentos AS
    SELECT 
        razao_social,
        cnpj,
        categoria_spend,
        objetoContrato,
        tipoContrato,
        dataVigenciaFim,
        valorGlobal,
        DATE_DIFF('day', CURRENT_DATE, dataVigenciaFim) AS dias_para_vencimento,
        CASE 
            WHEN DATE_DIFF('day', CURRENT_DATE, dataVigenciaFim) < 0 THEN 'VENCIDO'
            WHEN DATE_DIFF('day', CURRENT_DATE, dataVigenciaFim) <= 30 THEN 'CRÍTICO (0-30 dias)'
            WHEN DATE_DIFF('day', CURRENT_DATE, dataVigenciaFim) <= 90 THEN 'ATENÇÃO (30-90 dias)'
            WHEN DATE_DIFF('day', CURRENT_DATE, dataVigenciaFim) <= 180 THEN 'PLANEJAMENTO (90-180 dias)'
            ELSE 'REGULAR'
        END AS status_prazo
    FROM contratos
    WHERE dataVigenciaFim IS NOT NULL
    ORDER BY dias_para_vencimento ASC
""")

print("Views criadas:")
print("  - vw_contratos")
print("  - vw_sancoes") 
print("  - vw_contratos_risco")
print("  - vw_resumo_categoria_spend")
print("  - vw_resumo_mensal")
print("  - vw_kpis")
print("  - vw_concentracao_hhi")
print("  - vw_anomalias_preco")
print("  - vw_alerta_vencimentos")

In [46]:
# ============================================
# VERIFICAÇÃO DAS VIEWS
# ============================================

print("=== vw_kpis ===")
display(con.execute("SELECT * FROM vw_kpis").df())

print("\n=== vw_resumo_categoria_spend ===")
display(con.execute("SELECT * FROM vw_resumo_categoria_spend").df())

print("\n=== vw_contratos_risco (apenas com sanção) ===")
display(con.execute("SELECT * FROM vw_contratos_risco WHERE tem_sancao = true").df())

=== vw_kpis ===


,total_contratos,valor_total_contratos,total_fornecedores,fornecedores_com_sancao,percentual_risco
0,122,6.599479e+08,101,1,0.99



=== vw_resumo_categoria_spend ===


,categoria_spend,qtd_contratos,valor_total,valor_medio
0,Engenharia e Obras,14,5.821595e+08,4.158282e+07
1,Outros / Serviços Gerais,43,3.265972e+07,7.595285e+05
2,TI & Software,9,1.389841e+07,1.544268e+06
3,Logística - Locação/Serviço,4,7.992878e+06,1.998220e+06
4,Facilities - Limpeza/Zeladoria,4,6.515950e+06,1.628987e+06
5,Material de Escritório/Marketing,5,5.823525e+06,1.164705e+06
6,Facilities - Serviços Gerais,13,4.325933e+06,3.327641e+05
7,Eventos e Alimentação,4,3.992924e+06,9.982310e+05
8,Manutenção de Equipamentos,3,8.186000e+05,2.728667e+05
9,Serviços Profissionais,21,6.976086e+05,3.321946e+04



=== vw_contratos_risco (apenas com sanção) ===


,dataAssinatura,dataVigenciaInicio,dataVigenciaFim,cnpj,razao_social,unidadeOrgao,objetoContrato,valorGlobal,tipoContrato,categoria_spend,tem_sancao,lista_sancoes,lista_orgaos
0,2024-12-17,2024-12-17,2025-12-17,46344050000197,SUL AGUA EQUIPAMENTOS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,AQUISIÇÃO DE BENS DIVERSOS PARA ATENDIMENTO DE...,3649.86,Empenho,Outros / Serviços Gerais,True,Impedimento/proibição de contratar com prazo d...,"SEMASA, EPAGRI - Empresa de Pesquisa Agropecuá..."


In [47]:
display(con.execute("SELECT * FROM vw_contratos").df())

,dataAssinatura,dataVigenciaInicio,dataVigenciaFim,cnpj,razao_social,unidadeOrgao,objetoContrato,valorGlobal,tipoContrato,categoria_spend
0,2023-01-24,2023-01-24,2023-12-31,10719671000160,ELDEX DISTRIBUIDORA DE JORNAIS E REVISTAS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,PRESTAÇÃO DE SERVIÇOS DE FORNECIMENTO DE ASSIN...,10575.00,Empenho,TI & Software
1,2023-01-24,2023-01-24,2023-12-31,72579105000158,JOSE DOS REIS CHAVEIRO,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,"SERVIÇOS DE CHAVEIRO, COM FORNECIMENTO DE MATE...",4000.00,Empenho,Facilities - Serviços Gerais
2,2023-01-24,2023-01-24,2023-12-31,01319181000186,LAVANDERIA CRISTAL SERVICOS EXPRESSOS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,"SERVIÇOS DE LAVANDERIA, LAVAGEM, SECAGEM, PASS...",2000.00,Empenho,Facilities - Serviços Gerais
3,2023-01-24,2023-01-24,2023-12-31,24212365000148,EDMAR FERREIRA DA SILVA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,"SERVIÇOS DE DESINSETIZAÇÃO, DESRATIZAÇÃO, DESC...",3909.99,Empenho,Facilities - Limpeza/Zeladoria
4,2023-04-27,2023-04-27,2023-12-31,17067013000180,EGP SERVICOS LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA CONF...,3198.00,Empenho,Outros / Serviços Gerais
...,...,...,...,...,...,...,...,...,...,...
117,2025-10-28,2025-10-28,2027-10-28,51311014000167,LAG LICENCIAMENTOS DE SOFTWARES LTDA,SUBSECRETARIA DE TECNOLOGIA E INOVAÇAO/MME,"ASSINATURA CORPORATIVA DO SERVIÇO ON-LINE ""LIS...",4784.00,Empenho,Outros / Serviços Gerais
118,2025-11-28,2025-12-01,2025-12-05,04451208000188,COBUCCI DESENVOLVIMENTO HUMANO LTDA,COORD.GERAL DE RECURSOS HUMANOS/M.M.E,CURSO SOBRE “MEDIAÇÃO DE CONFLITOS E TÉCNICA D...,17210.00,Empenho,Serviços Profissionais
119,2025-12-23,2025-12-23,2028-12-23,59104760000604,TOYOTA DO BRASIL LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,AQUISIÇÃO DE 2 (DOIS) VEÍCULOS AUTOMOTORES NOV...,425429.00,Contrato (termo inicial),Logística - Aquisição de Frota
120,2025-12-02,2026-01-01,2027-01-01,28993675000106,O2 AMBIENTAL LTDA,MME-CGC-COORD.GERAL DE RECURSOS LOGISTICOS/DF,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA PARA PRES...,94080.00,Contrato (termo inicial),Facilities - Serviços Gerais


In [48]:
# Fechar conexão
con.close()

print("Banco DuckDB salvo em: data/gold/supply_chain.duckdb")

Banco DuckDB salvo em: data/gold/supply_chain.duckdb
